# Interpretability Analysis

Dedicated evidence notebook for model interpretation. It reads artifacts generated by the linear and tree pipelines and does not retrain models or recompute fitted estimators.

## Interpretability Scope

This notebook treats coefficients, impurity importance, permutation importance, SHAP, and partial dependence as fitted-model diagnostics. They support cautious interpretation of predictive behavior, not causal mechanism claims. The target remains environmentally related patenting share, and all interpretations should be read against the persistence baseline and latest-period test gap documented in the modeling notebooks.

In [ ]:
# ----- Project setup -----
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import Image, Markdown, display


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "3_models" / "scripts" / "model_config.py").exists():
            return candidate
    raise FileNotFoundError(f"Could not locate project root from {start}.")


ROOT = _find_project_root(Path(os.getcwd()).resolve())
SCRIPTS = ROOT / "3_models" / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from model_config import (  # noqa: E402
    COEFFICIENTS_OUTPUT,
    ERROR_BY_TARGET_QUANTILE_OUTPUT,
    ERROR_BY_YEAR_OUTPUT,
    FIGURE_INDEX_OUTPUT,
    FIGURES_DIR,
    TARGET_CORRELATIONS_OUTPUT,
    TOP_ERRORS_OUTPUT,
    TREE_FIGURE_INDEX_OUTPUT,
    TREE_FIGURES_DIR,
    TREE_HISTORICAL_DELTA_OUTPUT,
    TREE_IMPORTANCE_OUTPUT,
    TREE_OUTPUT_DIR,
    TREE_PARTIAL_DEPENDENCE_OUTPUT,
)

PAPER_FIGURES_DIR = ROOT / "4_analysis" / "figures" / "paper"
PERMUTATION_IMPORTANCE_OUTPUT = TREE_OUTPUT_DIR / "tree_model_permutation_importance.csv"
PERMUTATION_IMPORTANCE_FIGURE = PAPER_FIGURES_DIR / "fig7_permutation_importance.png"


def project_path(path_value) -> Path:
    path = Path(path_value)
    return path if path.is_absolute() else ROOT / path


def read_csv_or_empty(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def show_png(path: Path) -> None:
    path = project_path(path)
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"Missing figure: {path}")


def show_indexed_figure(index_path: Path, figure: str, fallback: Path) -> None:
    index = read_csv_or_empty(index_path)
    if not index.empty and {"figure", "path"}.issubset(index.columns):
        matches = index[index["figure"] == figure]
        if not matches.empty:
            show_png(project_path(matches.iloc[0]["path"]))
            return
    show_png(fallback)


print(f"Project root: {ROOT}")


## Artifact Availability

The report is intentionally artifact-first. If a row is missing, regenerate the corresponding modeling pipeline before interpreting the section.

In [ ]:
artifacts = [
    ("linear_coefficients", COEFFICIENTS_OUTPUT),
    ("linear_figure_index", FIGURE_INDEX_OUTPUT),
    ("tree_historical_baseline_delta", TREE_HISTORICAL_DELTA_OUTPUT),
    ("tree_importance", TREE_IMPORTANCE_OUTPUT),
    ("tree_figure_index", TREE_FIGURE_INDEX_OUTPUT),
    ("tree_partial_dependence", TREE_PARTIAL_DEPENDENCE_OUTPUT),
    ("tree_permutation_importance", PERMUTATION_IMPORTANCE_OUTPUT),
    ("linear_top_errors", TOP_ERRORS_OUTPUT),
    ("linear_error_by_year", ERROR_BY_YEAR_OUTPUT),
    ("linear_error_by_target_quantile", ERROR_BY_TARGET_QUANTILE_OUTPUT),
]
display(pd.DataFrame([
    {"artifact": name, "path": str(path.relative_to(ROOT) if path.exists() else path), "exists": path.exists()}
    for name, path in artifacts
]))


## Performance Boundary

Interpretability is only useful inside the model's demonstrated predictive boundary. The tree model explanations below are bounded by the historical-persistence comparison; if the selected model trails the country last-observed baseline, explanatory claims should remain descriptive and model-internal.

In [ ]:
historical_delta = read_csv_or_empty(TREE_HISTORICAL_DELTA_OUTPUT)
if not historical_delta.empty:
    keep = [
        "selected_model",
        "baseline_model",
        "selected_mae",
        "baseline_mae",
        "delta_mae_selected_minus_baseline",
        "selected_beats_baseline",
        "professor_interpretation",
    ]
    display(historical_delta.loc[:, [column for column in keep if column in historical_delta]].round(4))


## Global Importance

Linear coefficients and tree impurity importance give two complementary views. Coefficients are easiest to inspect for direction in the selected linear model, while tree importance summarizes split-driven predictive contribution and can be biased toward high-variance continuous predictors.

In [ ]:
show_indexed_figure(FIGURE_INDEX_OUTPUT, "coefficients", FIGURES_DIR / "linear_model_coefficients.png")
coefficients = read_csv_or_empty(COEFFICIENTS_OUTPUT)
if not coefficients.empty:
    # Filter active predictors with abs_coefficient > 1e-12.
    nonzero_coefficients = coefficients[coefficients["abs_coefficient"] > 1e-12].copy()
    if nonzero_coefficients.empty:
        print("Selected linear model has no nonzero coefficients above tolerance.")
    else:
        display(
            nonzero_coefficients.sort_values("abs_coefficient", ascending=False)
            .loc[:, ["feature", "coefficient", "abs_coefficient"]]
            .head(10)
            .round(4)
        )

show_indexed_figure(TREE_FIGURE_INDEX_OUTPUT, "importance", TREE_FIGURES_DIR / "tree_model_feature_importance.png")
tree_importance = read_csv_or_empty(TREE_IMPORTANCE_OUTPUT)
if not tree_importance.empty:
    display(tree_importance.loc[:, ["feature", "importance"]].head(10).round(4))


## SHAP Summary

SHAP is used here as a tree-model diagnostic for relative contribution patterns. It should be interpreted as model explanation, not as proof that a predictor causally changes environmental patenting.

In [ ]:
show_indexed_figure(
    TREE_FIGURE_INDEX_OUTPUT,
    "shap_summary",
    TREE_FIGURES_DIR / "tree_model_shap_summary.png",
)
print("Artifact key: tree_model_shap_summary")


## Permutation Importance

Permutation importance is the robustness counterpart to impurity importance. It is used here as a post-hoc test-block diagnostic, not for model selection or variable discovery. It asks how much predictive performance deteriorates when a feature is shuffled, so it is useful for checking whether the split-based ranking is overly model-internal.

In [ ]:
show_png(PERMUTATION_IMPORTANCE_FIGURE)
print("Artifact key: fig7_permutation_importance")
permutation_importance = read_csv_or_empty(PERMUTATION_IMPORTANCE_OUTPUT)
if not permutation_importance.empty:
    display(permutation_importance.head(10).round(4))


## Partial Dependence

Partial dependence shows the fitted tree model's average response across train+validation reference rows for top impurity-ranked predictors. It is a response diagnostic only. In correlated country-panel data, PDP can average over feature combinations that are sparse or absent in the observed data.

In [ ]:
show_indexed_figure(
    TREE_FIGURE_INDEX_OUTPUT,
    "partial_dependence",
    TREE_FIGURES_DIR / "tree_model_partial_dependence.png",
)
partial_dependence = read_csv_or_empty(TREE_PARTIAL_DEPENDENCE_OUTPUT)
if not partial_dependence.empty:
    display(partial_dependence.round(4).head(15))


## Cross-Model Consistency

This section compares the top-ranked predictors across coefficient magnitude, tree impurity importance, and permutation importance. Treat agreement as descriptive stability across diagnostics, not as a formal discovery claim.

In [ ]:
top_lists = []
if not coefficients.empty:
    # Filter active predictors with abs_coefficient > 1e-12.
    nonzero_coefficients = coefficients[coefficients["abs_coefficient"] > 1e-12].copy()
    top_lists.append(
        nonzero_coefficients.sort_values("abs_coefficient", ascending=False)
        .loc[:, ["feature", "abs_coefficient"]]
        .head(8)
        .rename(columns={"abs_coefficient": "score"})
        .assign(diagnostic="linear_abs_coefficient")
    )
if not tree_importance.empty:
    top_lists.append(
        tree_importance.loc[:, ["feature", "importance"]]
        .head(8)
        .rename(columns={"importance": "score"})
        .assign(diagnostic="tree_impurity_importance")
    )
if not permutation_importance.empty:
    top_lists.append(
        permutation_importance.loc[:, ["feature", "permutation_importance_mean"]]
        .head(8)
        .rename(columns={"permutation_importance_mean": "score"})
        .assign(diagnostic="tree_permutation_importance")
    )

if top_lists:
    consistency = pd.concat(top_lists, ignore_index=True)
    display(consistency.loc[:, ["diagnostic", "feature", "score"]].round(4))
    display(
        consistency.groupby("feature", as_index=False)
        .agg(diagnostic_count=("diagnostic", "nunique"))
        .sort_values(["diagnostic_count", "feature"], ascending=[False, True])
    )


## Error-Aware Interpretation

Interpretability should be read together with failure modes. If a predictor appears important but errors concentrate in specific years, countries, or target ranges, the explanation is conditional on those weaknesses.

In [ ]:
show_indexed_figure(FIGURE_INDEX_OUTPUT, "top_absolute_errors", FIGURES_DIR / "linear_model_top_absolute_errors.png")
top_errors = read_csv_or_empty(TOP_ERRORS_OUTPUT)
if not top_errors.empty:
    keep = [column for column in ["country_name", "year", "observed", "prediction", "absolute_error"] if column in top_errors]
    display(top_errors.loc[:, keep].head(12).round(4))

show_indexed_figure(FIGURE_INDEX_OUTPUT, "error_by_year", FIGURES_DIR / "linear_model_error_by_year.png")
error_by_year = read_csv_or_empty(ERROR_BY_YEAR_OUTPUT)
if not error_by_year.empty:
    display(error_by_year.round(4))

error_by_quantile = read_csv_or_empty(ERROR_BY_TARGET_QUANTILE_OUTPUT)
if not error_by_quantile.empty:
    display(error_by_quantile.round(4))


## Reviewer Caveats

- These diagnostics explain fitted prediction behavior, not causal policy effects.
- Tree impurity importance can favor continuous or high-variance predictors; permutation importance and SHAP are included as robustness checks.
- PDP curves can be off-manifold in correlated country-panel data, so read them as qualitative response diagnostics.
- Cross-model agreement is descriptive stability, not variable-selection proof.
- Any explanatory claim should be bounded by the persistence result: external predictors do not reliably beat national historical persistence on the latest test block.